In [1]:
import hashlib
import os
import platform
import uuid
# 修改为自己的学号
STUDENT_ID = 20244080114
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] pyexp
[学号]       20244080114
[机器指纹] e49d5bb589bc0a69
[RUN_ID]   20244080114-e49d5bb589bc0a69


### 字符串的乐趣

In [2]:
# 1. 判断回文 is_palindrome：忽略空格、不区分大小写
def is_palindrome(s):
    # 删除全部空格，转小写
    clean = s.replace(" ", "").lower()
    return clean == clean[::-1]

# 2. is_abecedarian 判断字母是否按顺序，忽略空格
def is_abecedarian(s):
    clean = s.replace(" ", "").lower()
    # 遍历一次，比较相邻字符
    for i in range(len(clean)-1):
        if clean[i] > clean[i+1]:
            return False
    return True

#3. remove_vowels 删除元音 a e i o u，大小写都处理
def remove_vowels(s):
    vowels = {"a","e","i","o","u","A","E","I","O","U"}
    res = ""
    for c in s:
        if c not in vowels:
            res += c
    return res

# 测试
print(is_palindrome("T A C O cat"))   # True
print(is_palindrome("level"))         # True
print(is_palindrome("hello"))         # False

print(is_abecedarian("abcd efgh xyz")) # True
print(is_abecedarian("adder"))         # True
print(is_abecedarian("dog"))           # False

print(remove_vowels("cat"))     # ct
print(remove_vowels("audio"))   # d

True
True
False
True
True
False
ct
d


### 列表的乐趣

In [3]:
#1 list_reverse，返回反转列表，如果输入不是列表抛错误
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError("参数必须是列表")
    return lst[::-1]

#2 is_sorted，序列升序，只遍历一次
def is_sorted(seq):
    for i in range(len(seq)-1):
        if seq[i] > seq[i+1]:
            return False
    return True

#测试
print(list_reverse([1,2,3])) # [3,2,1]
print(is_sorted([1,2,3,4])) # True
print(is_sorted([1,3,2]))   # False

[3, 2, 1]
True
False


### 作为向量的元组

In [4]:
#1 标量乘元组向量 vec_scalar_mult
def vec_scalar_mult(t, s):
    if not isinstance(t, tuple):
        raise TypeError("第一个参数必须是元组")
    if not isinstance(s, (int, float)):
        raise TypeError("标量s必须是int或者float")
    for elem in t:
        if not isinstance(elem, (int, float)):
            raise TypeError("元组内元素必须为数字")
    return tuple(x * s for x in t)

#2 向量内积 vec_inner_product
def vec_inner_product(t1, t2):
    if not isinstance(t1, tuple) or not isinstance(t2, tuple):
        raise TypeError("输入必须是元组")
    if len(t1) != len(t2):
        raise ValueError("两个向量长度不一致")
    total = 0.0
    for a,b in zip(t1,t2):
        total += a * b
    return total

#3 check_valid_mx 判断元组的元组是否合法矩阵
def check_valid_mx(m):
    if not isinstance(m, tuple):
        return False
    #外层每一个必须是元组
    for row in m:
        if not isinstance(row, tuple):
            return False
    #所有行长度相等
    row_len = len(m[0])
    for row in m:
        if len(row) != row_len:
            return False
        #每个元素必须数字
        for elem in row:
            if not isinstance(elem, (int, float)):
                return False
    return True

#4 mx_vec_mult 矩阵乘向量；矩阵是元组的元组，向量元组；矩阵 × 向量
def mx_vec_mult(mx, vec):
    if not check_valid_mx(mx):
        raise TypeError("不是合法矩阵")
    if not isinstance(vec, tuple):
        raise TypeError("向量必须是元组")
    n_col = len(mx[0])
    if len(vec) != n_col:
        raise ValueError("矩阵列数和向量维度不匹配")
    out = []
    for row in mx:
        out.append(vec_inner_product(row, vec))
    return tuple(out)

#测试
print(vec_scalar_mult((1,2,3), 2)) # (2,4,6)
print(vec_inner_product((1,2),(3,4))) #1*3 +2*4 =11.0

mx_test = ((1,2),(3,4))
print(check_valid_mx(mx_test))
print(mx_vec_mult(mx_test, (5,6)))

(2, 4, 6)
11.0
True
(17.0, 39.0)


### 更多向量的乐趣

##### 问题2：维度和元素应是类属性还是实例属性？说明选择的理由。
维度、向量元素应当作为实例属性。
理由：每一个Vector实例代表一个独立向量，不同实例可以拥有不同维度与元素；
如果作为类属性，所有对象共享同一套维度与元素，多个向量实例之间会互相覆盖，不符合向量定义。


In [5]:
class Vector:
    def __init__(self, dimension, elements=None):
        # 维度检查
        if not isinstance(dimension, int):
            raise TypeError("维度必须是整数")
        if dimension < 0:
            raise ValueError("维度不能为负数")
        # 元素默认：全0
        if elements is None:
            elements = [0.0]*dimension
        if len(elements) != dimension:
            raise ValueError("元素个数和维度不匹配")
        # 实例属性
        self.dimension = dimension
        self.elements = tuple(elements)

    # dot内积方法
    def dot(self, other):
        if not isinstance(other, Vector):
            raise TypeError("参数必须是Vector对象")
        if self.dimension != other.dimension:
            raise ValueError("向量维度不一致，无法计算内积")
        s = 0.0
        for a,b in zip(self.elements, other.elements):
            s += a*b
        return s

    # 相等判断 __eq__
    def __eq__(self, other):
        if not isinstance(other, Vector):
            return False
        if self.dimension != other.dimension:
            raise ValueError("维度不同不能比较相等")
        return self.elements == other.elements

    # 小于 字典序比较
    def __lt__(self, other):
        if not isinstance(other, Vector):
            raise TypeError("只能和Vector实例比较")
        if self.dimension != other.dimension:
            raise ValueError("维度不同不能比较大小")
        return self.elements < other.elements

    def __le__(self, other):
        return self < other or self == other

    def __gt__(self, other):
        return not self <= other

    def __ge__(self, other):
        return not self < other

    # 乘法运算符重载 __mul__，处理 v * x
    def __mul__(self, other):
        # 标量乘
        if isinstance(other, (int, float)):
            new_elems = [e*other for e in self.elements]
            return Vector(self.dimension, new_elems)
        # vector * vector：逐元素相乘
        elif isinstance(other, Vector):
            if self.dimension != other.dimension:
                raise ValueError("向量维度不一致")
            new_elems = [a*b for a,b in zip(self.elements, other.elements)]
            return Vector(self.dimension, new_elems)
        else:
            raise TypeError("不支持的乘法类型")

    # __rmul__ 处理 2 * v 标量在左边
    def __rmul__(self, other):
        return self * other

    # 方便打印看结果
    def __repr__(self):
        return f"Vector({self.dimension}, {list(self.elements)})"

# 测试
v1 = Vector(2, (2,4))
v2 = Vector(2, (2,5))
print(v1 < v2)
print(v1.dot(v2))
print(v1 * 3)
print(3 * v1)
print(v1 * v2)


True
24.0
Vector(2, [6, 12])
Vector(2, [6, 12])
Vector(2, [4, 20])
